# Yijun Bot — 一键训练 + 转 GGUF + 下载

## 用法
1. 上传这个 .ipynb 到 Colab（File → Upload notebook）
2. **Runtime → Change runtime type → A100 GPU**（或 T4，速度差别 5×）
3. 把 `finetune_clean.jsonl` 上传到 Drive 的 `MyDrive/yijun_bot/finetune_clean.jsonl`
4. 修改下面 Cell 1 的 `MODEL_SIZE`（`"3b"` 或 `"7b"`）
5. 顺序执行所有 cell

## 注意
Cell 4 后**必须手动 Runtime → Restart session**（不是 Disconnect！），不然 unsloth 不生效。

## Cell 1 — 配置

In [ ]:
MODEL_SIZE = "7b"  # 改成 "3b" 训 3B

# 一般不用动
REPO_URL = "https://github.com/KrimsonSun/Yijun.skill.git"
REPO_DIR = "/content/Yijun.skill"
DRIVE_DIR = "/content/drive/MyDrive/yijun_bot"

assert MODEL_SIZE in ("3b", "7b"), f"MODEL_SIZE must be 3b or 7b, got {MODEL_SIZE}"
print(f"Will train: yijun-{MODEL_SIZE}")

## Cell 2 — 检查 GPU

如果没有 GPU，回去 Runtime → Change runtime type → A100 GPU 后重新跑。

In [ ]:
!nvidia-smi | head -3
print()
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True)
assert r.returncode == 0, "❌ no GPU detected — change runtime type to GPU first"
print("✅ GPU detected")

## Cell 3 — 挂 Drive + 拉代码

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

assert os.path.exists(f"{DRIVE_DIR}/finetune_clean.jsonl"), (
    f"❌ {DRIVE_DIR}/finetune_clean.jsonl 不存在，先把数据集传到 Drive"
)
print("✅ repo + data ready")

## Cell 4 — 装 unsloth

不要手动 pip install torch！用 unsloth 的 colab-new 整套依赖，避免昨天那种 torch 版本冲突。

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## ⚠️ 现在手动重启 session

**Runtime → Restart session**（不是 Disconnect and delete runtime！）

重启后从 Cell 5 接着跑。Cell 1-4 已经做的事情（git clone、pip install）会保留。

## Cell 5 — 重启后重新挂 Drive + 验证 CUDA

In [ ]:
MODEL_SIZE = "7b"  # 重启后变量丢了，再设一遍。和 Cell 1 保持一致
REPO_DIR = "/content/Yijun.skill"
DRIVE_DIR = "/content/drive/MyDrive/yijun_bot"

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "❌ CUDA 不可用，runtime 没切到 GPU 或者 unsloth 装坏了"
print("✅ ready to train")

## Cell 6 — 训练（A100 上 3B ~3 min, 7B ~7-10 min）

In [ ]:
!rm -rf /content/yijun-{MODEL_SIZE} /content/sft_dataset
!bash {REPO_DIR}/colab/run.sh {MODEL_SIZE}

## Cell 7 — 转 GGUF Q4_K_M（~3-5 min）

结果输出到 `/content/yijun-{MODEL_SIZE}-Q4_K_M.gguf`：
- 3B：~1.9 GB
- 7B：~4.5 GB

In [ ]:
!bash {REPO_DIR}/colab/build_gguf.sh {MODEL_SIZE}
!ls -lh /content/yijun-{MODEL_SIZE}-Q4_K_M.gguf

## Cell 8 — 下载 GGUF 到本地

也可以从左侧文件树右键 → Download，对大文件更稳。

In [ ]:
from google.colab import files
files.download(f'/content/yijun-{MODEL_SIZE}-Q4_K_M.gguf')

## 完事

下载完后到 Mac 跑：

```bash
llama-cli \
    -m ~/Downloads/yijun-7b-Q4_K_M.gguf \
    -sys "$(cat ~/Documents/Git/Yijun.skill/prompts/yijun_voice_intimate.md)" \
    --temp 0.8 \
    --repeat-penalty 1.1 \
    -cnv
```

M4 Pro 跑 7B Q4 大概 30-50 tok/s，秒回。